<a id="inicio"></a>
<br><br><br>
<div align="left">
<h1><font size=6>Pandas</font></h1>
    <h2><font color="#5E5E5E" size=5>15. Índices de varios niveles </font> </h2>
<br><br><br><br>
</div>


<div align="right">
<font color="#B51700" size=4><b>Análisis y visualización de datos</b></font><br><br>
<font  size=3>Máster Universitario en Ingeniería Informática</font><br>
<font  size=3>Máster Universitario en Big Data y Computación en la Nube</font><br>
<font size=3>Máster Universitario en Inteligencia Artificial</font><br><br>
<font color="#5E5E5E" size=3><b>Luis de la Ossa  &nbsp;&nbsp;</b></font><span style="background-color:#B51700; color:white; padding:2px 6px; font-family:Arial, sans-serif;font-weight:bold;font-size:18px;">UCLM</span>
</div>

---

In [ ]:
import pandas as pd
pd.__version__

En esta libreta se utilizará el archivo `county.txt`, con datos sobre el censo de Estados Unidos.

In [ ]:
df_county = pd.read_csv('data/county.txt', sep='\t')
df_county.head()

---

*pandas* permite utilizar varios niveles de indexación, tanto para filas como para columnas. En este tutorial se describen los conceptos necesarios para el uso más común de estas estructuras. Se puede encontrar más información al respecto en la documentación [**( + inf )**](https://pandas.pydata.org/pandas-docs/stable/user_guide/advanced.html). 


Cuando se utilizan *varias columnas* (generalmente dos) como índice del `DataFrame`, se crea un índice de varios niveles que se organizan de **forma jerárquica**. Este tipo de índice se denomina también multiíndice.

In [ ]:
df_county.set_index(['state','name'], inplace=True)
df_county.head(10)

<div class="alert alert-block alert-info">

<i class="fa fa-info-circle" aria-hidden="true"></i> Si se pasa una lista de columnas a `index_col` al leer un `DataFrame` con `pd.read_csv()`, también crea un índice de varios niveles.
</div>

Cuando se utiliza un índice a varios niveles, el acceso natural a filas se hace mediante **tuplas** cuyo tamaño corresponde al número de índices, y con los valores del índice en cada nivel.

In [ ]:
df_county.loc[('Alabama','Bibb County')]          # Accede a una fila. Es recomendable utilizar una tupla.
#df_county.loc['Alabama','Bibb County']           # Se permite esta notación, pero puede inducir errores.

In [ ]:
#df_county.loc[['Alabama','Bibb County'], ['pop2000', 'pop2010']] # No funciona, porque utiliza una lista en lugar de una tupla.
df_county.loc[('Alabama','Bibb County'), ('pop2000', 'pop2010')]  

<div class="alert alert-block alert-warning">

<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
__Importante__: Hablamos de varios niveles porque pueden ser más de dos (aunque rara vez se da esta circunstancia). 
</div>

No es necesario especificar valores en todos los niveles para localizar elementos. Es posible omitir los valores ___a partir de___ un nivel. 

In [ ]:
df_county.loc['Alabama']
#df_county.loc['Alabama', ['pop2000', 'pop2010']]

Los índices de varios niveles permiten _slicing_ sobre el nivel principal.

In [ ]:
df_county.loc['Alabama':'California'].tail()

Es posible ordenar el `DataFrame` en función del índice jerárquico. En ese caso, la jerarquía se utiliza también en la ordenación.

In [ ]:
df_county.sort_index(inplace=False)

Cuando el `DataFrame` **está ordenado**, es posible hacer slicing con tuplas.

In [ ]:
#df_county.loc[('Alabama','Coosa County'):('Wyoming','Carbon County')] # No está ordenado y devuelve error
df_county.sort_index().loc[('Alabama','Coosa County'):('Wyoming','Carbon County')].head()

También se puede ordenar el `DataFrame` según un nivel determinado  del índice.

In [ ]:
df_county.sort_index(level=1, inplace=True)
df_county.head()

In [ ]:
df_county.sort_index(inplace=True) # Restaura el orden

Mediante la función `DataFrame.xs()` se puede acceder a los datos especificando el nivel del índice. Esto permite la indexación simple con índices en cualquier nivel.

In [ ]:
df_county.xs('Park County', level=1) # Existen tres condados con el mismo nombre, en diferentes estados.
#df_county.xs('Park County', level=1, drop_level=False) # Preserva el nivel
#df_county.xs('Park County', level="name") # Equivalente

Existen varios métodos que permiten crear un índice de varios niveles [**( + inf )**](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.MultiIndex.html).  Por ejemplo, el índice de varios niveles puede crearse a través de una lista de tuplas. En este ejemplo, se crea para un índice de dos niveles para las columnas.

In [ ]:
# Crea la lista de tuplas
level1 = ['population', 'population', 'money', 'money', 'money','money','money','money']
level2 = df_county.columns
tuples = list(zip(level1,level2))
#tuples

# Crea el índice de dos niveles
mi_columns = pd.MultiIndex.from_tuples(tuples, names=['ámbito', 'indicador'])
#mi_columns

# Establece un índice de dos niveles en las columnas.
df_county.columns = mi_columns
df_county.head()

<div class="alert alert-block alert-info">

<i class="fa fa-exclamation-circle" aria-hidden="true"></i>
Existen otros métodos como `from_arrays()`, `from_product()` o `from_frame()`.
</div>

El método `DataFrame.swaplevel()` permite intercambiar dos niveles en la jerarquía de índices. Utiliza el parámetro `axis` para determinar sobre qué índice actúa. No acepta el parámetro `inplace`, pero **devuelve una copia** del `DataFrame`.

In [ ]:
df_county.swaplevel(i=1,j=0, axis=1).head() # Cambia el índice de columnas

In [ ]:
# df_county.head() # No altera el DataFrame original

Otra posibilidad es reordenarlos con `DataFrame.reorder_levels()`. Las filas mantienen su posición.

In [ ]:
df_county.reorder_levels(['name','state']).head()

Al igual que ocurre con las filas, se puede acceder a las columnas individualmente, mediante tuplas. 

In [ ]:
df_county[('population','pop2000')].head()
# df_county['population','pop2000'].head() # Funciona, pero no se recomienda

También se pueden omitir los valores del índice a partir de un nivel.

In [ ]:
df_county['population'].head()

Para acceder a una columna en el nivel más bajo, hay que utilizar, como en el caso de las filas, `DataFrame.xs()`.

In [ ]:
df_county.xs('pop2000', level='indicador', axis=1).head() 

Puede accederse a los elementos mediante el uso de varias (en este caso dos) tuplas. 

In [ ]:
df_county.loc[('Alabama','Autauga County'), ('population')]
#df_county.loc[('Alabama','Autauga County'), ('population','pop2000')]

También pueden accederse a los elementos  a través del nivel (o niveles) más bajos del índice. 

In [ ]:
df_county.loc['Alabama', 'money'].head()

La función `DataFrame.groupby()` también puede aplicarse a índices de varios niveles. Acepta un parámetro, denominado `level`, que permite agrupar los datos según el valor del índice en un nivel (se vio anteriormente). 

In [ ]:
grupos_df = df_county.groupby(level=0)
for grupo, datos in grupos_df:
    print(grupo)
    display(datos.head())
    break;    # Solamente el primer grupo.

Otro ejemplo. Agrupa por nombre de condado, y muestra los grupos que más elementos tienen.

In [ ]:
df_county.groupby(level=1).size().nlargest(5)

### <font color="#5E5E5E"> Otros métodos de utilidad  </font>

Es posible crear un índice con uno de los niveles mediante `Index.get_level_values(nivel)`. 

In [ ]:
df_county.index.get_level_values('name')

A veces es más cómodo mostrar los índices de varios niveles como tuplas. Es posible hacerlo fijando la opción `display.multi_sparse` de *pandas* a `False`

In [ ]:
pd.set_option('display.multi_sparse', False)
df_county.head()

In [ ]:
pd.set_option('display.multi_sparse', True)

<br>
<div align="right">
<a href="#inicio"><font size=5 color="#B51700"><i class="fa fa-arrow-circle-up" aria-hidden="true"></i></font></a>
</div>

---

<div align="right">
<font size=5 color="#B51700"><i class="fa fa-coffee" aria-hidden="true"></i></font>
</div>